In [0]:
%run "../00. Common/Config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"


In [0]:
results_df = spark.table(bronze_table)

In [0]:
display(results_df)

In [0]:
results_selected_df = results_df.drop("url")

In [0]:
results_renamed_df = (
    results_selected_df
        .withColumnsRenamed({
            "raceName": "race_name",
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "date": "race_date",
            "number": "car_number",
            "grid": "grid_position",
            "laps": "completed_laps",
            "position": "final_position",
            "positionText": "final_position_text" 
        })
)

In [0]:
import pyspark.sql.functions as F
results_valid_df = results_renamed_df.filter(
    F.col("season").isNotNull() &
    F.col("round").isNotNull() &
    F.col("constructor_id").isNotNull() &
    F.col("driver_id").isNotNull()
)

In [0]:
results_distinct_df = results_valid_df.dropDuplicates(["season", "round", "constructor_id", "driver_id"])

In [0]:
display(results_distinct_df)

In [0]:
import pyspark.sql.functions as F
results_final_df = (
    results_distinct_df
        .withColumn('race_name',F.initcap("race_name"))
)

In [0]:
(
    results_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))